## Step 0: Setup & Data Generation

In [ ]:
import numpy as np

from numcompute.io import load_csv, load_csv_chunks
from numcompute.preprocessing import SimpleImputer, StandardScaler, OneHotEncoder
from numcompute.pipeline import Pipeline
from numcompute.sort_search import stable_sort, topk, quickselect, binary_search
from numcompute.rank import rank, percentile
from numcompute.stats import mean, std, WelfordStats, histogram
from numcompute.optim import grad, jacobian

def create_dummy_csv(filename="demo_dataset.csv", num_rows=100):
    np.random.seed(42)
    
    feature1 = np.random.normal(50, 15, num_rows)  # mean 50, std 15
    feature2 = np.random.uniform(0, 1000, num_rows) # Between 0 ~ 1000, random number 
    feature3 = np.random.choice([0, 1, 2], size=num_rows) # categorical data, between 0,1,2
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write("feature_1,feature_2,feature_3\n")
        
        for i in range(num_rows):
            f1, f2, f3 = feature1[i], feature2[i], feature3[i]
            
            if i % 10 == 0:
                f1_str = "NaN"
            else:
                f1_str = f"{f1:.2f}"
                
            if i % 15 == 0:
                f2_str = "?"
            else:
                f2_str = f"{f2:.2f}"
                
            f.write(f"{f1_str},{f2_str},{f3}\n")
            
    print(f"Data successfully generated and saved to {filename}")

create_dummy_csv("demo_dataset.csv", num_rows=100)

Data successfully generated and saved to demo_dataset.csv


## Step 1: Data loading

In [3]:
chunked_data = load_csv_chunks(filepath="demo_dataset.csv",chunk_size=30, skip_header=1)

print("load data with chunking, shape of each chunk")
for data in chunked_data:
    print(data.shape)

print("\n" + "-" * 20 + "\n")

print("Last chunk's data")
print(data)


load data with chunking, shape of each chunk
(30, 3)
(30, 3)
(30, 3)
(10, 3)

--------------------

Last chunk's data
[[   nan    nan   0.  ]
 [ 64.53 568.31   1.  ]
 [ 39.47  93.67   1.  ]
 [ 45.09 367.72   2.  ]
 [ 44.12 265.2    1.  ]
 [ 28.05 243.99   2.  ]
 [ 54.44 973.01   0.  ]
 [ 53.92 393.1    0.  ]
 [ 50.08 892.05   0.  ]
 [ 46.48 631.14   0.  ]]


In [4]:
data = load_csv(filepath="demo_dataset.csv", skip_header=1)

# check 2D array shape
print(data.shape)

# check whether invalid values change into np.nan
print(np.isnan(data).sum(axis=0))


(100, 3)
[10  7  0]


## Step 2: Preprocessing and Pipeline

In [5]:
numeric_preprocessor = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ]
)

categorical_preprocessor = OneHotEncoder(drop_first=True)

numeric_data = data[:,:2]
categorical_data = data[:,2:]

preprocessed_numeric_data = numeric_preprocessor.fit_transform(numeric_data)
preprocessed_categorical_data = categorical_preprocessor.fit_transform(categorical_data)

print("=== numeric_preprocessor result ===")
print("Result Shape:", preprocessed_numeric_data.shape)
print("Number of NaN:", np.isnan(preprocessed_numeric_data).sum())
print("Mean of each column:", np.round(preprocessed_numeric_data.mean(axis=0), 4))
print("Std of each column:", np.round(preprocessed_numeric_data.std(axis=0), 4))

print("\n=== OneHotEncoder Result ===")
print("Original Column Shape:", categorical_data.shape)
print("Encoded Shape:", preprocessed_categorical_data.shape)
print("Raw Unique Categories:", np.unique(categorical_data))
print("Encoded Categories (drop_first=True):", categorical_preprocessor.categories_[0])
print("First 5 rows of encoded data:\n", preprocessed_categorical_data[:5])

data_prep = np.hstack([preprocessed_numeric_data, preprocessed_categorical_data])

=== numeric_preprocessor result ===
Result Shape: (100, 2)
Number of NaN: 0
Mean of each column: [-0.  0.]
Std of each column: [1. 1.]

=== OneHotEncoder Result ===
Original Column Shape: (100, 1)
Encoded Shape: (100, 2)
Raw Unique Categories: [0. 1. 2.]
Encoded Categories (drop_first=True): [1. 2.]
First 5 rows of encoded data:
 [[1. 0.]
 [0. 0.]
 [1. 0.]
 [0. 0.]
 [0. 1.]]


## Step 3: Sort, Search, and Rank

In [7]:
# scaled numeric feature 1, original categorical feature 3
num_col = data_prep[:, 0]
cat_col = categorical_data.flatten() # for testing rank()

print("=== Sort & Search ===")
# Top-K
top5_vals, top5_idx = topk(num_col, k=5, largest=True)
print("Top 5 values:", np.round(top5_vals, 4))
print("Top 5 indices:", top5_idx)

# Quickselect
kth_val = quickselect(num_col, k=10)
print("\n10th smallest value:", np.round(kth_val, 4))

# Binary Search
sorted_col = stable_sort(num_col)

# Binary Search 1, find exsisting value
target_val = sorted_col[37]
found_idx = binary_search(sorted_col, target_val)
print(f"Target value ({np.round(target_val, 4)}) found at index: {found_idx}")

# 2. Binary Search 2, find not exsisting value
missing_val = sorted_col[-1] + 10.0
missing_idx = binary_search(sorted_col, missing_val)
print(f"Missing value ({np.round(missing_val, 4)}) search result: {missing_idx}")

print("\n=== Rank & Percentile ===")
print("Original categories (first 10):", cat_col[:10])
print("Rank (method='average'):", rank(cat_col[:10], method='average'))
print("Rank (method='min'):    ", rank(cat_col[:10], method='dense'))

# Percentile
pcts = percentile(num_col, q=[25, 50, 75])
print("\n25th, 50th, 75th percentiles:", np.round(pcts, 4))

=== Sort & Search ===
Top 5 values: [2.2629 1.9523 1.9356 1.9052 1.8885]
Top 5 indices: [31  6 73 71  3]

10th smallest value: -1.3596
Target value (-0.1901) found at index: (37, True)
Missing value (12.2629) search result: (100, False)

=== Rank & Percentile ===
Original categories (first 10): [1. 0. 1. 0. 2. 0. 0. 0. 2. 2.]
Rank (method='average'): [6.5 3.  6.5 3.  9.  3.  3.  3.  9.  9. ]
Rank (method='min'):     [2. 1. 2. 1. 3. 1. 1. 1. 3. 3.]

25th, 50th, 75th percentiles: [-0.498  -0.      0.5483]


## Step 4: Statistics

In [9]:
# scaled numeric feature 1
num_col = data_prep[:, 0]

print("=== Basic Stats ===")
print("Mean:", np.round(mean(num_col), 4))
print("Std: ", np.round(std(num_col), 4))

print("\n=== Welford's Streaming Stats ===")
# Use first half, and second half 
w_stats = WelfordStats()
w_stats.update(num_col[:50])
w_stats.update(num_col[50:])

print("Streaming Mean:", np.round(w_stats.mean_, 4))
print("Streaming Variance:", np.round(w_stats.variance, 4))

print("\n=== Histogram ===")
counts, edges = histogram(num_col, bins=5)
print("Counts per bin:", counts)
print("Bin edges:", np.round(edges, 4))

=== Basic Stats ===
Mean: -0.0
Std:  1.0

=== Welford's Streaming Stats ===
Streaming Mean: -0.0
Streaming Variance: 1.0101

=== Histogram ===
Counts per bin: [ 5 14 42 27 12]
Bin edges: [-2.8313 -1.8125 -0.7936  0.2252  1.244   2.2629]


## Step 5: Numerical Optimization

In [11]:
print("=== Numerical Differentiation (Finite Difference) ===")

# 1. test function, f(x, y) = x^2 + 3xy + y^2
def test_scalar_func(x):
    return x[0]**2 + 3*x[0]*x[1] + x[1]**2

x_point = np.array([1.0, 2.0])
grad_val = grad(test_scalar_func, x_point)

print("Function: f(x,y) = x^2 + 3xy + y^2")
print(f"Gradient at {x_point}:", np.round(grad_val, 4))
# expected answer: df/dx = 2x+3y = 8, df/dy = 3x+2y = 7 => [8.0, 7.0] 

# 2. test function g(x, y) = [x^2, x + y]
def test_vector_func(x):
    return np.array([x[0]**2, x[0] + x[1]])

jacob_val = jacobian(test_vector_func, x_point)
print("\nFunction: g(x,y) = [x^2, x+y]")
print(f"Jacobian at {x_point}:\n", np.round(jacob_val, 4))

=== Numerical Differentiation (Finite Difference) ===
Function: f(x,y) = x^2 + 3xy + y^2
Gradient at [1. 2.]: [8. 7.]

Function: g(x,y) = [x^2, x+y]
Jacobian at [1. 2.]:
 [[2. 0.]
 [1. 1.]]


## Step 6: Metrics & Benchmarking (Loop vs Vectorised)

In [15]:
import math
from numcompute.utils import euclidean_distance, softmax
from numcompute.metrics import accuracy, mse
from numcompute.benchmarking import compare_implementations

print("=== 1. Metrics Validation ===")
y_true_cls = np.array([0, 1, 1, 0, 1])
y_pred_cls = np.array([0, 1, 0, 0, 1])
print("Accuracy:", accuracy(y_true_cls, y_pred_cls))

y_true_reg = np.array([1.5, 2.0, 3.5])
y_pred_reg = np.array([1.0, 2.5, 3.5])
print("MSE:", mse(y_true_reg, y_pred_reg))


def mse_loop(y_true, y_pred):
    total = 0.0
    for a, p in zip(y_true, y_pred):
        total += (float(a) - float(p)) ** 2
    return total / len(y_true)

def euclidean_distance_loop(X, Y):
    distances = np.zeros((X.shape[0], Y.shape[0]))
    for i in range(X.shape[0]):
        for j in range(Y.shape[0]):
            dist = 0.0
            for k in range(X.shape[1]):
                dist = math.hypot(dist, float(X[i, k]) - float(Y[j, k]))
            distances[i, j] = dist
    return distances

def softmax_loop(x):
    max_val = float(max(x))
    exp_values = [math.exp(float(v) - max_val) for v in x]
    total = sum(exp_values)
    return np.array([v / total for v in exp_values])


print("\n=== 2. Short Performance Table (Loop vs NumPy) ===")
rng = np.random.default_rng(42)

# table header
print("-" * 65)
print(f"{'Operation':>20} | {'Samples':>10} | {'Loop (ms)':>10} | {'NumPy (ms)':>10} | {'Speedup':>8}")
print("-" * 65)

# Test 1: MSE (100,000 samples)
n_mse = 100000
y_t, y_p = rng.normal(size=n_mse), rng.normal(size=n_mse)
t_mse = compare_implementations(mse_loop, mse, y_t, y_p, repeat=3)
print(f"{'MSE':>20} | {n_mse:>10} | {t_mse['loop_time']*1000:>10.2f} | {t_mse['vec_time']*1000:>10.2f} | {t_mse['speedup']:>7.1f}x")

# Test 2: Euclidean Distance (100x100 matrix, 8 features)
n_euc = 100
X_euc, Y_euc = rng.normal(size=(n_euc, 8)), rng.normal(size=(n_euc, 8))
t_euc = compare_implementations(euclidean_distance_loop, euclidean_distance, X_euc, Y_euc, repeat=3)
print(f"{'Euclidean Dist':>20} | {n_euc:>10} | {t_euc['loop_time']*1000:>10.2f} | {t_euc['vec_time']*1000:>10.2f} | {t_euc['speedup']:>7.1f}x")

# Test 3: Softmax (100,000 samples)
n_soft = 100000
x_soft = rng.normal(loc=1000.0, scale=2.0, size=n_soft)
t_soft = compare_implementations(softmax_loop, softmax, x_soft, repeat=3)
print(f"{'Softmax':>20} | {n_soft:>10} | {t_soft['loop_time']*1000:>10.2f} | {t_soft['vec_time']*1000:>10.2f} | {t_soft['speedup']:>7.1f}x")
print("-" * 65)

=== 1. Metrics Validation ===
Accuracy: 0.8
MSE: 0.16666666666666666

=== 2. Short Performance Table (Loop vs NumPy) ===
-----------------------------------------------------------------
           Operation |    Samples |  Loop (ms) | NumPy (ms) |  Speedup
-----------------------------------------------------------------
                 MSE |     100000 |      13.48 |       0.14 |    94.7x
      Euclidean Dist |        100 |      17.56 |       1.19 |    14.8x
             Softmax |     100000 |      12.67 |       0.37 |    34.1x
-----------------------------------------------------------------
